# 04 - Downstream analysis

Once the model is a `cobra.Model`, everything cobrapy does is available. Rough timings on the
2584-reaction model with Gurobi:

| | | |
|---|---|---|
| FBA | `model.optimize()` | < 1 s |
| pFBA | `pfba` | 1-2 s |
| FVA | `flux_variability_analysis` | 1-3 min |
| single gene deletion | `single_gene_deletion` | 1-3 min |
| single reaction deletion | `single_reaction_deletion` | 2-5 min |
| double gene deletion | `double_gene_deletion` | quadratic, subset first |
| sampling | `cobra.sampling.sample` | very slow |

`rf.optimize_cb_model` only exists because rFASTCORMICS needs an FBA internally; for anything
else use cobrapy.

Section 9 checks FBA and gene deletion against the MATLAB reference.

## Load

In [ ]:
import numpy as np
import pandas as pd
import cobra
from cobra.flux_analysis import (pfba, flux_variability_analysis,
                                 single_gene_deletion, single_reaction_deletion,
                                 double_gene_deletion, find_essential_genes)
import rfastcormics_v2 as rf

SOLVER = "gurobi"        # "glpk" ships with cobrapy if no licence

# A: straight from notebook 03, if it is still in memory
# ctx = rf.to_cobrapy(BCRAmodel, "BRCA_context_specific")

# B: read back what notebook 03 exported
ctx = cobra.io.read_sbml_model("BRCA_context_specific_model.xml")
ctx.solver = SOLVER

print(ctx)
print(f"{len(ctx.reactions)} rxns / {len(ctx.metabolites)} mets / {len(ctx.genes)} genes")
print("objective:", ctx.objective.expression)

In [ ]:
if not any(r.objective_coefficient for r in ctx.reactions):
    ctx.objective = "biomass_reaction"     # SBML round-trip sometimes loses it

In [ ]:
import pickle
def load_generic(path="data/model.pkl"):
    m = pickle.load(open(path, "rb"))
    return rf.CobraModel(
        S=m["S"], rxns=m["rxns"], mets=m["mets"], lb=m["lb"], ub=m["ub"], c=m["c"],
        b=m["b"], csense=m["csense"], osense=m["osense"], genes=m["genes"],
        rules=m["rules"], grRules=m["grRules"], rxnGeneMat=m["rxnGeneMat"],
        subSystems=m["subSystems"], metFormulas=m["metFormulas"],
        metNames=m["metNames"], rxnNames=m["rxnNames"])

control = rf.to_cobrapy(rf.change_objective(load_generic(), "biomass_reaction"), "control")
control.solver = SOLVER
control

## 1. FBA

In [ ]:
sol = ctx.optimize()
print("status:", sol.status)
print("growth:", sol.objective_value)

flux = sol.fluxes[sol.fluxes.abs() > 1e-9].sort_values(key=abs, ascending=False)
pd.DataFrame({"flux": flux.head(15),
              "name": [ctx.reactions.get_by_id(i).name for i in flux.head(15).index]})

In [ ]:
print("context-specific:", ctx.slim_optimize())
print("control         :", control.slim_optimize())

## 2. pFBA

FBA's optimum is not unique (same reason FASTCORE drifts between solvers). pFBA minimises the
total flux at the optimal growth rate, which gives a unique and more physiological solution.

In [ ]:
pfba_sol = pfba(ctx)
print("growth:", pfba_sol.fluxes["biomass_reaction"])
print("total flux:", pfba_sol.fluxes.abs().sum())
print("active reactions:", int((pfba_sol.fluxes.abs() > 1e-9).sum()), "/", len(ctx.reactions))

## 3. FVA

`fraction_of_optimum=0.9` allows growth to drop to 90% of the optimum, which is the usual
setting. Use `1.0` for the strict optimum. 2 LPs per reaction, so this takes a few minutes.

In [ ]:
fva = flux_variability_analysis(ctx, fraction_of_optimum=0.9, processes=4)
fva["range"] = fva["maximum"] - fva["minimum"]
print(fva.shape)
fva.head()

In [ ]:
tol = 1e-9
blocked  = fva[(fva["minimum"].abs() < tol) & (fva["maximum"].abs() < tol)]
fixed    = fva[(fva["range"] < tol) & (fva["maximum"].abs() >= tol)]
flexible = fva.sort_values("range", ascending=False)

print("blocked:", len(blocked))
print("flux fixed to a single value:", len(fixed))
flexible.head(10)

In [ ]:
# exchange reactions only - uptake / secretion ranges
ex_ids = [r.id for r in ctx.exchanges]
fva_ex = flux_variability_analysis(ctx, reaction_list=ex_ids, fraction_of_optimum=0.9)
fva_ex["range"] = fva_ex["maximum"] - fva_ex["minimum"]
fva_ex.sort_values("minimum").head(15)     # most negative minimum = largest uptake

## 4. Single gene deletion

In [ ]:
wt = ctx.slim_optimize()
ko = single_gene_deletion(ctx)
ko["gene"] = ko["ids"].apply(lambda s: list(s)[0])
ko["ratio"] = ko["growth"] / wt
ko = ko[["gene", "growth", "ratio", "status"]].sort_values("ratio")
print("wild type:", wt)
print("essential (ratio < 0.01):", int((ko["ratio"] < 0.01).sum()))
ko.head(15)

Note this deletes one gene ID at a time. MATLAB's `singleGeneDeletion(..., uniqueGene=1)`
groups transcripts of the same gene - see section 9 for the version that matches it.

In [ ]:
dico = pd.read_csv("data/dico.csv", dtype=str)
entrez2symbol = dict(zip(dico["ENTREZ"], dico["SYMBOL"]))

ko["entrez"] = ko["gene"].str.replace(r"\.\d+$", "", regex=True)
ko["symbol"] = ko["entrez"].map(entrez2symbol)
ko.head(15)

In [ ]:
essential = find_essential_genes(ctx, threshold=1e-6)
essential_ids = sorted(g.id for g in essential)
essential_sym = sorted({entrez2symbol.get(g.split(".")[0], g) for g in essential_ids})
print(len(essential_ids), "essential genes")
print(essential_sym[:40])

Genes essential in the tumour model but not in the generic model are the selective
candidates - this is what the rFASTCORMICS paper does for drug target prediction.

In [ ]:
essential_ctl = {g.id for g in find_essential_genes(control, threshold=1e-6)}
essential_ctx = set(essential_ids)

only_ctx = sorted(essential_ctx - essential_ctl)
print(f"essential in context: {len(essential_ctx)}   in control: {len(essential_ctl)}")
print(f"context only: {len(only_ctx)}")
pd.DataFrame({"gene": only_ctx,
              "symbol": [entrez2symbol.get(g.split(".")[0], "") for g in only_ctx]}).head(30)

## 5. Single reaction deletion

In [ ]:
ko_rxn = single_reaction_deletion(ctx)
ko_rxn["rxn"] = ko_rxn["ids"].apply(lambda s: list(s)[0])
ko_rxn["ratio"] = ko_rxn["growth"] / wt
ko_rxn = ko_rxn[["rxn", "growth", "ratio", "status"]].sort_values("ratio")
print("essential reactions:", int((ko_rxn["ratio"] < 0.01).sum()))
ko_rxn.head(10)

## 6. Double gene deletion

Quadratic - 1290 genes is 830k pairs. Subset first: genes whose single knockout hurts but
doesn't kill.

In [ ]:
candidates = ko.loc[(ko["ratio"] > 0.01) & (ko["ratio"] < 0.95), "gene"].tolist()
print(len(candidates), "candidates ->", len(candidates) * (len(candidates) - 1) // 2, "pairs")

candidates = candidates[:40]          # start small, then widen
dko = double_gene_deletion(ctx, gene_list1=candidates, processes=4)
dko["ratio"] = dko["growth"] / wt
dko = dko[dko["ids"].apply(len) == 2].sort_values("ratio")
dko.head(15)

In [ ]:
# synthetic lethal: neither single knockout kills, the pair does
single_ratio = dict(zip(ko["gene"], ko["ratio"]))
def is_synthetic_lethal(row):
    a, b = sorted(row["ids"])
    return (row["ratio"] < 0.01 and
            single_ratio.get(a, 1) > 0.5 and single_ratio.get(b, 1) > 0.5)

sl = dko[dko.apply(is_synthetic_lethal, axis=1)]
print("synthetic lethal pairs:", len(sl))
sl.head(10)

## 7. Changing the medium

`model.medium` is a `{exchange reaction: max uptake}` dict. Wrap in `with model:` so it reverts.

In [ ]:
print("open uptake reactions:", len(ctx.medium))
list(ctx.medium.items())[:10]

In [ ]:
with ctx:
    medium = ctx.medium
    if "EX_glc_D[e]" in medium:
        medium["EX_glc_D[e]"] = 0.0
        ctx.medium = medium
        print("without glucose:", ctx.slim_optimize())
    else:
        print("different exchange id, look for:", [k for k in medium if "glc" in k])

print("restored:", ctx.slim_optimize())

## 8. Export

In [ ]:
fva.to_csv("results_fva.csv")
ko.to_csv("results_single_gene_deletion.csv", index=False)
ko_rxn.to_csv("results_single_reaction_deletion.csv", index=False)
pd.Series(pfba_sol.fluxes, name="pfba_flux").to_csv("results_pfba_fluxes.csv")

## 9. Check against MATLAB

`workspaceExample.mat` also stores what the second half of `tutorialScript.m` produced:

| | |
|---|---|
| `fbaControl.f` / `fbaBCRA.f` | FBA objective for both models |
| `grRateWTControl` / `grRateWTBCRA` | wild-type growth (753.336 / 50.672) |
| `grRatioControl` (1884x1) / `grRatioBCRA` (1108x1) | relative growth after each gene knockout |

Two levels: (a) run on MATLAB's own `BCRAmodel` with its bounds, which takes the FASTCORE
difference out of the picture and only tests the analysis code; (b) run on the model rebuilt
here, which is the end-to-end question.

In [ ]:
ref = np.load("data/matlab_reference.npz", allow_pickle=True)
print("FBA        control:", ref["fba_f_control"], " BRCA:", ref["fba_f_BRCA"])
print("grRateWT   control:", ref["grRateWT_control"], " BRCA:", ref["grRateWT_BRCA"])
print("deletions  control:", ref["grRatio_control"].size, "genes / BRCA",
      ref["grRatio_BRCA"].size, "genes")

### 9a. Rebuild MATLAB's BCRAmodel

In [ ]:
generic = load_generic()
keep = set(map(str, ref["BCRAmodel_rxns"]))
sub, _ = rf.remove_rxns(generic, [str(r) for r in generic.rxns if str(r) not in keep])
sub = rf.remove_unused_genes(sub)

order = {str(r): i for i, r in enumerate(ref["BCRAmodel_rxns"])}
idx = np.array([order[str(r)] for r in sub.rxns])
sub.lb, sub.ub = ref["BCRAmodel_lb"][idx], ref["BCRAmodel_ub"][idx]
sub = rf.change_objective(sub, "biomass_reaction")

print("rebuilt:", sub.n_rxns, "rxns /", sub.n_mets, "mets /", len(sub.genes), "genes")
print("matlab :", ref["BCRAmodel_rxns"].size, "rxns / 2134 mets /",
      ref["BCRAmodel_genes"].size, "genes")

cb_ml = rf.to_cobrapy(sub, "MATLAB_BCRAmodel"); cb_ml.solver = SOLVER
print(f"\nFBA:  python {cb_ml.slim_optimize():.8f}   matlab {ref['grRateWT_BRCA']:.8f}")

Should print `50.67169641` on both sides. If it doesn't, stop here - something is wrong
with the rebuild and nothing below will mean anything.

### 9b. Gene deletion, and a `strmatch` trap in COBRA

MATLAB runs `singleGeneDeletion(model, 'FBA', [], 0, 1)`. The last argument is `uniqueGene`,
and that branch of `singleGeneDeletion.m` does:

```matlab
[geneList, rem] = strtok(geneList, '.');   % 8639.1 -> 8639
geneList = unique(geneList);               % sorted
...
delGenes = model.genes(strmatch(geneList{i}, model.genes));
```

`strmatch` is a **prefix** match. Knocking out gene `26` also knocks out `26.1`, `260.1`,
`2645.1`, `26191.1` and so on. The shorter the ENTREZ ID, the more genes get taken out with it.

Run both groupings and see which one reproduces MATLAB:

In [ ]:
from cobra.manipulation import knock_out_model_genes


def matlab_single_gene_deletion(cb_model, match="prefix"):
    """Replicate MATLAB singleGeneDeletion(..., uniqueGene=1).

    Use match="prefix" to reproduce MATLAB's strmatch behavior.
    Use match="exact" to group genes by their base ENTREZ ID.
    """
    ids = [g.id for g in cb_model.genes]
    base = sorted({i.split(".")[0] for i in ids})

    wt = cb_model.slim_optimize()
    growth = []

    for b in base:
        if match == "prefix":
            grp = [i for i in ids if i.startswith(b)]
        elif match == "exact":
            grp = [i for i in ids if i.split(".")[0] == b]
        else:
            raise ValueError("match must be 'exact' or 'prefix'")

        with cb_model:
            knock_out_model_genes(cb_model, grp)
            growth.append(cb_model.slim_optimize(error_value=0.0))

    g = np.array(growth, dtype=float)
    return np.array(base, dtype=object), g, g / wt, wt
    base = sorted({i.split(".")[0] for i in ids})          # strtok + unique
    wt = cb_model.slim_optimize()
    growth = []
    for b in base:
        grp = [i for i in ids if (i.startswith(b) if match == "prefix" else i.split(".")[0] == b)]
        with cb_model:
            knock_out_model_genes(cb_model, grp)
            growth.append(cb_model.slim_optimize(error_value=0.0))
    g = np.array(growth, float)
    return np.array(base, dtype=object), g, g / wt, wt


ml_ratio = ref["grRatio_BRCA"]
for mode in ("exact", "prefix"):
    base, ko_g, ratio, wt_ml = matlab_single_gene_deletion(cb_ml, match=mode)
    bad = ~np.isclose(ratio, ml_ratio, atol=1e-6, rtol=1e-6)
    print(f"{mode:6s}: differing genes {bad.sum():4d}/{len(base)}   "
          f"max|diff| = {np.nanmax(np.abs(ratio - ml_ratio)):.2e}   "
          f"essential python={int((ratio < 0.01).sum())} matlab={int((ml_ratio < 0.01).sum())}")
    if bad.any():
        print("        ", list(base[bad])[:12])

What I get:

```
exact : differing genes   18/1108   max|diff| = 1.00e+00   essential python=41 matlab=54
         ['109','114','115','13','18','19','219','229','2592','30','39','48']
prefix: differing genes    0/1108   max|diff| = 4.75e-14   essential python=54 matlab=54
```

So the prefix match reproduces `grRatioBCRA` to floating-point precision, which confirms the
FBA + GPR evaluation + knockout logic here is correct. It also means **13 of MATLAB's 54
"essential genes" are artefacts of the prefix match**. Every affected gene is a 1-4 digit
ENTREZ ID, i.e. a prefix of many longer ones.

Use `match="exact"` for actual target screening. Use `match="prefix"` only to line up with
published MATLAB numbers, and say so if you do. This is a `singleGeneDeletion.m` problem, not
an rFASTCORMICS one - it hits any model with transcript-suffixed gene IDs and `uniqueGene=1`.

### 9c. End-to-end

Now with the model rebuilt here, so the ~1% reaction difference is included. Reaction
membership is the wrong thing to compare; growth rate and essential genes are the right ones.

In [ ]:
base_p, ko_p, ratio_p, wt_p = matlab_single_gene_deletion(ctx,   match="exact")
base_m, ko_m, ratio_m, wt_m = matlab_single_gene_deletion(cb_ml, match="exact")

print(f"growth   python {wt_p:.6f}   matlab {wt_m:.6f}   "
      f"relative diff {abs(wt_p - wt_m) / wt_m * 100:.4f}%")

ess_p = set(base_p[ratio_p < 0.01])
ess_m = set(base_m[ratio_m < 0.01])
print(f"essential   python {len(ess_p)}   matlab {len(ess_m)}   "
      f"intersection {len(ess_p & ess_m)}   jaccard {len(ess_p & ess_m) / len(ess_p | ess_m):.3f}")
print("python only:", sorted(ess_p - ess_m))
print("matlab only:", sorted(ess_m - ess_p))

## 10. Sampling

Gives the distribution of feasible fluxes rather than one point, but 2500+ reactions takes
tens of minutes to hours for 1000 samples.

```python
from cobra.sampling import sample
samples = sample(ctx, n=1000, method="optgp", processes=4)
samples.describe().T.sort_values("mean")
```

Try `n=100` first to gauge the speed.